<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_104_residual_stream_interventions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧠 **Module 104 — Residual-Stream Interventions** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 15 · Mechanistic Interpretability Workshop (Cohen ML4LLM deep-dives)


# Module 104 — Residual-Stream Interventions

> M101 looked at tokens, M102 at embedding geometry, M103 at the output head. **This module enters the model itself.** Cohen Ch5 (projects 24-32) treats every **layer's residual-stream activations** as data — across-layer similarity, category selectivity, the "current layer = previous + adjustment" framing that defines the residual stream, **effective dimensionality**, and the two foundational mech-interp interventions: **logit lens** and **activation patching for IOI** (Indirect Object Identification). By the end you'll know how to read a model's *intermediate beliefs* and how to *causally* manipulate them.

### What you'll cover
1. The **residual stream** — what it actually is
2. **Cosine sim within and across layers** — when do features "lock in"?
3. **Category selectivity** — animals, locations, emotions, syntax
4. **Layer = previous + adjustment** — the LayerNorm + residual algebra
5. **Layer-specific noise and scaling** — robustness experiments
6. **Effective dimensionality** of hidden states (PCA / SVD)
7. **Hidden state dimensionality reduction** for downstream tasks
8. **Sentiment-decision trees on hidden states** — probing alternative
9. **Logit lens** (nostalgebraist 2020) — read each layer's "best guess"
10. **Activation patching for IOI** (Wang 2023) — the causal-intervention gold standard


## 1 · The residual stream — what it actually is

Modern Transformers (decoder-only, M19/M20) compute:

```
h^{(l+1)} = h^{(l)} + Attn^{(l)}(LayerNorm(h^{(l)}))
                  + MLP^{(l)} (LayerNorm(h^{(l+1)/2}))
```

Each layer **adds to** the running hidden state rather than replacing it. The hidden state is called the **residual stream** because every layer's output is a *residual update*. Anthropic's 2022 "Transformer Circuits Thread" coined the modern mech-interp framing:

```
   Token embed  ─►  residual stream (R^D, T positions)
                   │
   Layer 1 attn  ───►  write to residual
   Layer 1 MLP   ───►  write to residual
   ...
   Layer L attn  ───►  write to residual
   Layer L MLP   ───►  write to residual
                   │
   LayerNorm  ─►  W_out  ─►  logits
```

**Two consequences for mech-interp:**
1. **The residual stream is a communication channel.** Each layer reads from it (via LayerNorm) and writes to it (additively).
2. **You can read the "running belief"** at any layer by passing it through the final `W_out` and softmax — the **logit lens** (Section 9).

> 📐 **The mental model that makes everything click.** Think of the residual stream as a **stack of registers** that all layers read/write simultaneously. Some directions encode tokens; some encode syntax; some encode entities; some encode "is this a question?" Each layer fires when its input pattern matches and adds its own contribution. Mech-interp is the art of figuring out *which direction encodes what*.


## 2 · Across-layer cosine similarity

Cohen proj 24-25 picks a set of stimuli (animals, vehicles, emotions) and asks: **how does the hidden state for the same token change as it progresses through layers?**

```
h_cat^{(0)}, h_cat^{(1)}, ..., h_cat^{(L)}
              ↓
cos(h_cat^{(l)}, h_cat^{(l+1)})    for l = 0, 1, ..., L-1
```

Typical pattern (Llama-3-8B):

```
Layer  0→1   1→2   2→3   ...  10→11  11→12  ...   30→31
cos    0.98  0.92  0.88  ...  0.85   0.80   ...   0.95
```

**Three regimes** that show up across most models:
- **Early layers (0-3)**: small changes — tokens are being contextualized
- **Middle layers (4-N/2)**: largest changes — heavy semantic processing
- **Late layers (N-3..N)**: small changes again — predictions stabilize

Category-level sim across layers reveals **when "cat" stops looking like "dog" and starts looking like "cat"** — typically around layer 12 of 32 for Llama-3. This is the **representational refinement** view of Transformer depth.

> 🧠 **Why early layers are conservative.** A 2024 finding: in deep nets, early layers compute *position-aware token features* that downstream layers need verbatim. Modifying them disrupts everything. Mid-layer ablations are much more recoverable.


## 3 · Category selectivity

Cohen proj 25 extends cosine sim to **category selectivity** — for layer `l`, compute the average within-category cosine `S_in` and average between-category cosine `S_out`. Selectivity = `(S_in − S_out) / (S_in + S_out)`.

```
Selectivity   ↑
       1.0    │            ╭──────────╮
              │           ╱            ╲
       0.5    │         ╱                ╲
              │       ╱                    ╲___
       0.0    │_____╱____________________________
                0   4   8  12  16  20  24  28  32   layer
                early   middle              late
```

Three categories peak at different layers (Llama-3-8B):
- **Token identity** (cat vs dog) — peaks at layer 4-6
- **Semantic categories** (animals vs vehicles) — peaks at layer 12-18
- **Sentiment** (joy vs sadness) — peaks at layer 20-28

**Practical implication.** When you build a probe (M103) or extract embeddings (M30 RAG), choosing the right *layer* matters as much as the right *model*. Sentence-BGE uses layer ~24 of 28 (one before the head); reranking can benefit from layer ~16. Production embedding models report the layer they pool from.


## 4 · Layer = previous + adjustment — the residual algebra

Cohen proj 26 makes the residual update explicit:

```
delta_l = h^{(l+1)} - h^{(l)}     # what THIS layer added

magnitude:   ||delta_l|| / ||h^{(l)}||           # relative update size
angle:       arccos(<delta_l, h^{(l)}>/||·||·||·||)  # direction of update
```

Empirically (Llama-3-8B, on natural text):
- `||delta_l|| / ||h^{(l)}|| ≈ 0.05-0.15` — each layer **adds a small fraction** of the running state
- Update angle ≈ 60-80° — adjustments are *mostly orthogonal* to the running state

**Two crucial insights from the residual algebra:**
1. **Magnitude grows superlinearly** — early `||h||` ≈ 1 (post-init), late `||h||` ≈ 8-15 (Llama-3-8B). Each layer adds about 5-10% of the running magnitude.
2. **The residual stream is "free-form"** — there's no mandatory direction; layers compete for the same `D` dimensions. This is the **superposition** that makes Sparse Autoencoders (M100 callback) meaningful.


In [ ]:
# Cohen proj 26 — residual updates per layer
import torch
from transformers import AutoModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
m   = AutoModel.from_pretrained("gpt2", output_hidden_states=True).eval()
text = "The Eiffel Tower is in Paris."
ids  = tok(text, return_tensors="pt").input_ids
with torch.no_grad():
    H = torch.stack(m(ids).hidden_states)  # [L+1, B, T, D]
# focus on last token
H_last = H[:, 0, -1, :]                    # [L+1, D]
mags = H_last.norm(dim=-1)                 # [L+1] — magnitude per layer
deltas = (H_last[1:] - H_last[:-1]).norm(dim=-1)
print("magnitudes:", mags.round(decimals=2))
print("relative updates:", (deltas / mags[:-1]).round(decimals=3))


## 5 · Layer-specific noise + scaling

Cohen proj 27: a **mini-experiment** to probe robustness. Insert noise at a chosen layer and see how output perplexity changes.

```
For each layer l ∈ {0, ..., L}:
    h^{(l)} ← h^{(l)} + ε·η       where η ~ N(0, I)
    Run rest of model.
    Measure perplexity / accuracy on a test set.
```

You discover:
- **Early layers are sensitive** — small `ε` causes large degradation
- **Middle layers are robust** — can take ε up to ~20% of the magnitude before noticeable degradation
- **Last 2-3 layers are sensitive again** — they're directly read by `W_out`

**Scaling experiments** (Cohen proj 27 cont.) try multiplying `h^{(l)}` by a scalar `α`:
- `α=0` (zeroing the layer's contribution) often breaks the model entirely
- `α=2` (doubling) is surprisingly tolerated for many layers
- The **U-shape** is real and consistent across architectures

This is the **first form of causal intervention** — modify a hidden state and measure the *downstream* change. Activation patching (Section 10) generalizes this.

> 🛡️ **Why this matters for quantization (M90 callback).** INT4/INT8 quantization adds *noise* to weights. The same U-shape predicts which layers are quantization-sensitive — early + final, and which can be aggressively quantized — middle. This is exactly why **GGUF k-quants** (Q4_K_M) use different bit-widths per layer.


## 6 · Effective dimensionality

Cohen proj 28 takes hidden states `[N × D]` (N tokens) for a corpus, computes the SVD, and asks: **how many singular values are "non-negligible"**?

Three metrics:
```
participation_ratio = (Σ σ_i)² / Σ σ_i²        # ~1 for one direction, ~D for uniform
effective_rank      = exp(H(p))   where p_i = σ_i² / Σ σ_j²
99%_dim             = smallest k s.t. Σ_{i≤k} σ_i² / Σ σ_j² ≥ 0.99
```

Typical findings:
- **Layer 0 (token embeddings)**: effective rank ~64 (compressed)
- **Middle layers**: effective rank ~256-512 (expanded for processing)
- **Last layer**: effective rank ~128 (compressed back for `W_out`)

The model **uses ~5-10% of its embedding dimension at the input/output** and **~30-50% in the middle**. This is the **hourglass-in-feature-space** that makes superposition possible.

> 📐 **Connection to LoRA (M28 callback).** LoRA works because the *update* to weights during fine-tuning has low effective rank — usually `r ≈ 8-64` is enough. The same low-rank structure that mech-interp finds is what makes parameter-efficient fine-tuning viable.


## 7 · Hidden state dimensionality reduction

Cohen proj 29 takes hidden states from many layers and reduces them to **2D / 3D for visualization** via PCA / t-SNE / UMAP.

| Reducer | Best for | Cost |
|---|---|---|
| **PCA** | Linear, reproducible, fast | Cheap |
| **t-SNE** | Local structure, clusters | Slow, non-reproducible |
| **UMAP** | Global + local; the modern default | Medium |
| **Linear probing axes** | Targeted: gender, sentiment, position | Need labels |

What you see when you UMAP-reduce hidden states from many sentences:
- **Topic clusters** form naturally
- **Sentence boundaries** appear as sharp jumps in trajectory
- **Token type** (noun, verb, adjective) clusters by layer
- **Layer-by-layer evolution** of a single token traces a path

**Atlas / Vector-Atlas / Datamap-PCA** (2024 visualization tools) are production grade for this — they let you scroll a corpus and watch which sentences cluster together by latent semantics.


## 8 · Sentiment decision trees on hidden states

Cohen proj 30 builds a small **decision tree** on hidden states to predict sentence sentiment. It's the **interpretable cousin** of the linear probe (M103).

```
hidden states (layer l, mean-pooled)  →  decision tree (depth 5)
                                       ↓
              if dim_42 > 0.3 and dim_117 < -0.1: positive
                                       ...
```

Insights from this experiment:
- **5-15 dimensions** are usually enough to classify sentiment with 80-90% accuracy
- The *important* dimensions cluster — meaning sentiment is a **direction**, not a basis-axis
- **Tree depth 3-5** is the sweet spot — deeper trees overfit

This connects to **Sparse Autoencoder** research (Anthropic 2024, OpenAI 2024) — find a basis of *interpretable* directions. The decision-tree approach is the cheap version of an SAE feature, useful for any classifier-style probing where you want to see the *rules*.

> 🌳 **Why trees over linear models for some probes.** Linear probes find a single hyperplane. Decision trees find **interaction effects** — "high dim-42 AND low dim-117". For probes of complex compositional features (negation, hedging, sarcasm), tree-based probes often outperform.


## 9 · Logit lens — reading the model's intermediate beliefs

Cohen proj 31 introduces the **logit lens** (nostalgebraist 2020): project the residual stream at every layer through the final `W_out` and softmax. The result is the **model's "best-guess token" at every layer**.

```
For each layer l ∈ {0, ..., L}:
    logits_l = W_out · LayerNorm(h^{(l)})    # use FINAL LayerNorm + W_out
    top_token_l = argmax(logits_l)
    top_prob_l  = max(softmax(logits_l))
```

**A typical logit-lens trajectory** for "The Eiffel Tower is in [predict]":

```
Layer  Top-token       p
  0    "the"           0.02     (token embed; uninformative)
  4    "France"        0.07     (early bias)
  8    "Paris"         0.12     (becomes correct early)
 12    "Paris"         0.34     (confidence growing)
 16    "Paris"         0.55     (consolidation)
 20    "Paris"         0.71
 24    "Paris"         0.78     (peak in middle-late)
 28    "Paris"         0.82
 32    "Paris"         0.89     (final answer)
```

**Findings from logit lens (across 2020-2026 mech-interp work):**
- **Correct token often appears at layer N/2** — middle layers know the answer; later layers refine confidence
- **Wrong tokens at early layers** are *biases* — the model leans toward common continuations before context evidence kicks in
- **Logit-lens "convictions"** can be edited via activation patching (Section 10)

> 🔭 **Why logit lens works at all.** It works because of **residual-stream linearity** — every layer writes to the same `D`-dim register; the final readout always projects through the *same* `W_out`. The intermediate hidden states already live in the "vocabulary space" of the model.

> ⚠️ **The tuned-lens upgrade** (Belrose 2023). Vanilla logit lens applies the *final* layer's LayerNorm at every depth, which is wrong (LayerNorm gain differs per layer). The **tuned lens** trains a per-layer translator that maps each layer's residual into the final-layer geometry. Used by the Anthropic interp team since 2023.


In [ ]:
# Logit lens in 15 lines
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
m   = AutoModelForCausalLM.from_pretrained("gpt2", output_hidden_states=True).eval()

text = "The Eiffel Tower is in"
ids  = tok(text, return_tensors="pt").input_ids
with torch.no_grad():
    out = m(ids)
hiddens = out.hidden_states                          # tuple of [B, T, D]
W_out   = m.lm_head.weight                            # [V, D]  (often tied with wte)
ln_f    = m.transformer.ln_f                          # final LayerNorm

for l, h in enumerate(hiddens):
    logits = W_out @ ln_f(h[0, -1, :])                # final-token logits via logit lens
    p, idx = torch.softmax(logits, -1).max(-1)
    print(f"layer {l:>2}: top='{tok.decode([idx])}' p={p:.3f}")


## 10 · Activation patching + IOI — causal intervention

Cohen proj 32 closes Ch5 with the **gold-standard mech-interp method**: **activation patching**, applied to the **IOI** (Indirect Object Identification) task.

**The IOI task** (Wang et al. 2023, "Interpretability in the Wild"):

```
"When Mary and John went to the store, John gave a drink to ___"
                                                      ↑
                                       model should predict "Mary"
```

The model has to **track that John gave to Mary** (not John). This is solved primarily by **3-5 attention heads** in middle layers — the "name-mover" and "S-inhibition" heads.

**Activation patching pipeline:**
```
1. Run model on a CLEAN prompt:    "John gave a drink to ___"   → predict "Mary"
2. Run model on a CORRUPTED prompt: "Mary gave a drink to ___"   → predict "John"
3. Re-run the CORRUPTED prompt, but PATCH activation at (layer l, position p, component)
   with the CLEAN version's activation
4. Measure: did the prediction flip back toward "Mary"?
```

If patching at `(l=12, position=John, head=9.6)` restores the answer, you've **causally** identified that head as part of the IOI circuit.

**The 2026 mech-interp toolkit:**

| Technique | What it does |
|---|---|
| **Activation patching** | Swap one activation; measure downstream impact |
| **Path patching** | Patch all paths *except* a target subgraph |
| **Attribution patching** (Syed 2023) | Linear approximation of patching; ~10× faster |
| **ACDC** (Conmy 2023) | Auto-discover the circuit with iterative patching |
| **Sparse Autoencoders (SAE)** (Anthropic 2024) | Decompose residual stream into interpretable features |
| **Steering vectors** | Add a direction (e.g., "refuse" direction) at inference |
| **Attribution graphs** | Visualize which earlier components caused a token |

> 🌟 **The intellectual capstone of Cohen Ch5.** Activation patching is **causation, not correlation**. A probe says "this layer encodes X." Patching says "remove X and the prediction stops working — X is *causally* used." Every modern interpretability claim (Anthropic Claude 3 paper, OpenAI Sparse Autoencoders 2024, DeepMind Gemma-Scope) backs claims with patching experiments.

## ✅ Recap

- The **residual stream** is the shared communication channel — every layer reads and writes additively.
- **Across-layer cosine** shows three regimes: small early changes → big middle changes → small late stabilization.
- **Category selectivity** peaks at different layers — token identity ~4-6, semantics ~12-18, sentiment ~20-28.
- **Layer = previous + small adjustment** — relative update ~5-15% of magnitude; mostly orthogonal direction.
- **Noise / scaling experiments** reveal a U-shape — early + last layers sensitive, middle robust.
- **Effective dimensionality** is hourglass-shaped: ~64 input, ~256-512 middle, ~128 final. Underwrites LoRA viability.
- **Logit lens** (and tuned lens) reads each layer's "best-guess token" — correct answer often appears at layer N/2.
- **Decision-tree probes** find interaction effects between residual-stream dimensions; SAEs generalize this.
- **Activation patching** + IOI is the **causal** mech-interp gold standard; ACDC / path patching / attribution patching scale it.
- **2026 toolkit**: TransformerLens, nnsight, Pyvene, SAE Lens, ActAdd / steering vectors.

Next: **M105 — Attention Mechanism Analysis** (Cohen Ch6).
